# DCE Results Metrics Aggregation

This notebook provides metrics aggregation functionality:
- **Metrics Aggregation**: Computes average metrics across seeds for each optimization method

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import json
from pathlib import Path
import re

# Configuration
DCE_RESULTS_PATH = "DCE_Results/cardio"  # Can be easily modified
CSV_OUTPUT_DIR = "metrics_aggregated"  # Directory for saving CSV results

print("🚀 DCE Results Metrics Aggregation")
print(f"📁 Results directory: {DCE_RESULTS_PATH}")
print(f"📊 CSV output directory: {CSV_OUTPUT_DIR}")

🚀 DCE Results Metrics Aggregation
📁 Results directory: DCE_Results/cardio
📊 CSV output directory: metrics_aggregated


## Metrics Aggregation Functions

Computes average metrics across seeds for each optimization method

In [2]:
# Global variables to track skipped experiments
SKIPPED_METRICS = []
MISSING_DATA_SUMMARY = {}

def extract_strategy_name(strategy_dir_name):
    """Extract simplified strategy name from directory name"""
    if "Bayesian" in strategy_dir_name:
        return "Bayesian"
    elif "DifferentialEvolution" in strategy_dir_name:
        return "Differential Evolution"
    elif "Genetic" in strategy_dir_name:
        return "Genetic"
    elif "MonteCarlo" in strategy_dir_name:
        return "Monte Carlo"
    elif "SimulatedAnnealing" in strategy_dir_name:
        return "Simulated Annealing"
    else:
        return "Unknown"

def extract_method_type(strategy_dir_name):
    """Extract method type from directory name"""
    if "cone_cat_" in strategy_dir_name:
        return "CAT"
    elif "cone_cont_" in strategy_dir_name:
        return "CONT"
    elif "original_all_" in strategy_dir_name:
        return "NONE"
    elif "cone_all_" in strategy_dir_name:
        return "ALL"
    else:
        return "Unknown"

def aggregate_metrics_for_experiment(experiment_path):
    """Aggregate metrics for all optimization methods in an experiment"""
    
    # Extract dataset name from path for data loader
    path_parts = Path(experiment_path).parts
    dataset_name = path_parts[-3] if len(path_parts) >= 3 else "unknown"
    
    # Find all optimization method directories - support multiple naming patterns
    method_dirs = []
    for d in os.listdir(experiment_path):
        if os.path.isdir(os.path.join(experiment_path, d)):
            # Check for various method directory patterns
            if any(pattern in d for pattern in ["cone_all_", "cone_cat_", "cone_cont_", "original_all_"]):
                method_dirs.append(d)
    
    if not method_dirs:
        print(f"⚠️ No optimization method directories found in: {experiment_path}")
        return None
    
    results = {}
    
    for method_dir in method_dirs:
        method_path = os.path.join(experiment_path, method_dir)
        strategy_name = extract_strategy_name(method_dir)
        method_type = extract_method_type(method_dir)
        
        # Combine strategy and method type for unique identification
        combined_name = f"{strategy_name} ({method_type})"
        
        # Find all seed directories in this method
        seed_dirs = [d for d in os.listdir(method_path) 
                    if os.path.isdir(os.path.join(method_path, d)) and d.startswith("seed_")]
        
        if not seed_dirs:
            print(f"⚠️ No seed directories found in: {method_path}")
            # Track methods without seeds
            model_name = path_parts[-2] if len(path_parts) >= 2 else "unknown"
            key = f"{dataset_name}_{model_name}"
            if key not in MISSING_DATA_SUMMARY:
                MISSING_DATA_SUMMARY[key] = {"dataset": dataset_name, "model": model_name, "missing_strategies": []}
            MISSING_DATA_SUMMARY[key]["missing_strategies"].append(f"{combined_name} (no seeds)")
            continue
        
        # Collect metrics from all seeds
        all_metrics = []
        missing_metrics_count = 0
        
        for seed_dir in seed_dirs:
            seed_path = os.path.join(method_path, seed_dir)
            metrics_file = os.path.join(seed_path, "metrics_summary.json")
            
            if os.path.exists(metrics_file):
                try:
                    with open(metrics_file, 'r') as f:
                        metrics = json.load(f)
                    
                    # Check if metrics file is empty or has no useful data
                    if not metrics or all(v is None or pd.isna(v) for v in metrics.values()):
                        print(f"⚠️ Empty/invalid metrics in: {metrics_file}")
                        missing_metrics_count += 1
                        continue
                    
                    all_metrics.append(metrics)
                except Exception as e:
                    print(f"❌ Error reading {metrics_file}: {e}")
                    missing_metrics_count += 1
            else:
                print(f"⚠️ metrics_summary.json not found in: {seed_path}")
                missing_metrics_count += 1
        
        if not all_metrics:
            print(f"⚠️ No valid metrics found for {combined_name}")
            # Track strategies with no valid metrics
            model_name = path_parts[-2] if len(path_parts) >= 2 else "unknown"
            key = f"{dataset_name}_{model_name}"
            if key not in MISSING_DATA_SUMMARY:
                MISSING_DATA_SUMMARY[key] = {"dataset": dataset_name, "model": model_name, "missing_strategies": []}
            reason = f"no valid metrics ({missing_metrics_count}/{len(seed_dirs)} seeds failed)"
            MISSING_DATA_SUMMARY[key]["missing_strategies"].append(f"{combined_name} ({reason})")
            continue
        
        # Calculate average metrics
        averaged_metrics = {}
        
        # Get all metric keys from all metrics dicts (in case some have different keys)
        all_metric_keys = set()
        for metrics in all_metrics:
            all_metric_keys.update(metrics.keys())
        
        for metric_key in sorted(all_metric_keys):
            # Collect values for this metric across all seeds
            values = []
            for metrics in all_metrics:
                if metric_key in metrics and not pd.isna(metrics[metric_key]):
                    values.append(metrics[metric_key])
            
            if values:
                avg_value = np.mean(values)
                
                # Apply formatting rules as specified
                if "Coverage Rate" in metric_key:
                    averaged_metrics[metric_key] = round(avg_value, 2)  # 2 decimal places
                elif "Percentile Difference" in metric_key:
                    averaged_metrics[metric_key] = round(avg_value, 2)  # 2 decimal places
                else:
                    averaged_metrics[metric_key] = round(avg_value, 4)  # 4 decimal places
            else:
                averaged_metrics[metric_key] = np.nan
        
        results[combined_name] = averaged_metrics
        print(f"✅ Processed {len(all_metrics)} seeds for {combined_name} (skipped {missing_metrics_count})")
    
    return results

def create_metrics_csv(experiment_path, aggregated_results):
    """Create CSV file with aggregated metrics"""
    if not aggregated_results:
        return None
    
    # Extract experiment name from path
    path_parts = Path(experiment_path).parts
    if len(path_parts) >= 3:
        # Format: dataset/model/DCE_params
        dataset = path_parts[-3]
        model = path_parts[-2] 
        dce_params = path_parts[-1]
        experiment_name = f"{dataset}_{model}_{dce_params}"
    else:
        experiment_name = "_".join(path_parts)
    
    # Replace problematic characters in filename
    safe_experiment_name = re.sub(r'[<>:"/\|?*=,()]', '_', experiment_name)
    
    # Get all metric names
    all_metrics = set()
    for strategy_metrics in aggregated_results.values():
        all_metrics.update(strategy_metrics.keys())
    all_metrics = sorted(list(all_metrics))
    
    # Create DataFrame
    strategy_names = list(aggregated_results.keys())
    
    # Create the CSV data with strategies as columns and metrics as rows
    csv_data = {}
    csv_data['Metric'] = all_metrics
    
    for strategy in strategy_names:
        csv_data[strategy] = [aggregated_results[strategy].get(metric, np.nan) for metric in all_metrics]
    
    df = pd.DataFrame(csv_data)
    df = df.set_index('Metric')
    
    # Ensure CSV output directory exists
    os.makedirs(CSV_OUTPUT_DIR, exist_ok=True)
    
    # Save CSV file
    csv_filename = f"{safe_experiment_name}.csv"
    csv_path = os.path.join(CSV_OUTPUT_DIR, csv_filename)
    
    df.to_csv(csv_path)
    print(f"✅ Saved metrics CSV: {csv_path}")
    
    return csv_path

def process_all_metrics(dataset_filter=None, model_filter=None):
    """Process all experiments and create metrics CSV files
    
    Args:
        dataset_filter (str): Filter by dataset name (e.g., 'german_credit')
        model_filter (str): Filter by model name (e.g., 'LightGBM')
    """
    global SKIPPED_METRICS, MISSING_DATA_SUMMARY
    SKIPPED_METRICS = []  # Reset tracking
    MISSING_DATA_SUMMARY = {}  # Reset tracking
    
    if not os.path.exists(DCE_RESULTS_PATH):
        print(f"❌ DCE_Results directory not found: {DCE_RESULTS_PATH}")
        return
    
    # Find all experiment directories (should contain DCE parameters)
    experiment_pattern = os.path.join(DCE_RESULTS_PATH, "**", "DCE_u1=*")
    experiment_dirs = glob.glob(experiment_pattern, recursive=True)
    
    # Filter by dataset and/or model
    filtered_experiment_dirs = []
    for exp_dir in experiment_dirs:
        path_parts = Path(exp_dir).parts
        if len(path_parts) >= 3:
            dataset_name = path_parts[-3]
            model_name = path_parts[-2]
            
            # Apply filters
            if dataset_filter and dataset_filter not in dataset_name:
                continue
            if model_filter and model_filter not in model_name:
                continue
                
        filtered_experiment_dirs.append(exp_dir)
    
    print(f"📊 Found {len(filtered_experiment_dirs)} experiment directories (filtered from {len(experiment_dirs)} total)")
    if dataset_filter:
        print(f"🔍 Dataset filter: {dataset_filter}")
    if model_filter:
        print(f"🔍 Model filter: {model_filter}")
    
    success_count = 0
    
    for exp_dir in filtered_experiment_dirs:
        print(f"\n🔍 Processing experiment: {exp_dir}")
        
        aggregated_results = aggregate_metrics_for_experiment(exp_dir)
        
        if aggregated_results:
            csv_path = create_metrics_csv(exp_dir, aggregated_results)
            if csv_path:
                success_count += 1
                
                # Display summary
                print(f"📋 Summary for {os.path.basename(exp_dir)}:")
                for strategy, metrics in aggregated_results.items():
                    print(f"   {strategy}: {len([v for v in metrics.values() if not pd.isna(v)])} metrics")
        else:
            # Track experiments with no results
            path_parts = Path(exp_dir).parts
            dataset_name = path_parts[-3] if len(path_parts) >= 3 else "unknown"
            model_name = path_parts[-2] if len(path_parts) >= 2 else "unknown"
            
            SKIPPED_METRICS.append({
                'dataset': dataset_name,
                'model': model_name,
                'experiment_path': exp_dir,
                'reason': 'No valid optimization results found'
            })
    
    print(f"\n✅ Successfully processed {success_count}/{len(filtered_experiment_dirs)} experiments")
    print(f"📁 All CSV files saved in: {CSV_OUTPUT_DIR}")
    if SKIPPED_METRICS:
        print(f"⚠️ Skipped {len(SKIPPED_METRICS)} experiments due to missing/invalid data")

print("📦 Metrics aggregation functions loaded successfully!")

📦 Metrics aggregation functions loaded successfully!


## Execute Metrics Aggregation and Create CSV Files

In [3]:
# Configuration - Set dataset and model filters here
DATASET_FILTER = None  # e.g., "german_credit" or None for all datasets
MODEL_FILTER = None    # e.g., "LightGBM" or None for all models

# Examples:
# DATASET_FILTER = "german_credit"  # Only process german_credit dataset
# MODEL_FILTER = "LightGBM"         # Only process LightGBM model
# Both can be used together to filter specific dataset-model combinations

print("🔧 Configuration:")
print(f"📊 Dataset filter: {DATASET_FILTER if DATASET_FILTER else 'All datasets'}")
print(f"🤖 Model filter: {MODEL_FILTER if MODEL_FILTER else 'All models'}")
print()

# Run metrics aggregation
print("📊 Starting Metrics Aggregation...")
process_all_metrics(dataset_filter=DATASET_FILTER, model_filter=MODEL_FILTER)
print("✅ Metrics aggregation completed!")

🔧 Configuration:
📊 Dataset filter: All datasets
🤖 Model filter: All models

📊 Starting Metrics Aggregation...
📊 Found 5 experiment directories (filtered from 5 total)

🔍 Processing experiment: DCE_Results/cardio\LightGBM\DCE_u1=0.5,u2=0.3,l=0.2,r=1.0,mi=200,tk=1,nt=10
✅ Processed 10 seeds for Genetic (ALL) (skipped 0)
✅ Processed 10 seeds for Monte Carlo (ALL) (skipped 0)
✅ Saved metrics CSV: metrics_aggregated\cardio_LightGBM_DCE_u1_0.5_u2_0.3_l_0.2_r_1.0_mi_200_tk_1_nt_10.csv
📋 Summary for DCE_u1=0.5,u2=0.3,l=0.2,r=1.0,mi=200,tk=1,nt=10:
   Genetic (ALL): 10 metrics
   Monte Carlo (ALL): 10 metrics

🔍 Processing experiment: DCE_Results/cardio\MLP\DCE_u1=0.5,u2=0.3,l=0.2,r=1.0,mi=200,tk=1,nt=10
✅ Processed 10 seeds for Genetic (ALL) (skipped 0)
✅ Processed 10 seeds for Monte Carlo (ALL) (skipped 0)
✅ Saved metrics CSV: metrics_aggregated\cardio_MLP_DCE_u1_0.5_u2_0.3_l_0.2_r_1.0_mi_200_tk_1_nt_10.csv
📋 Summary for DCE_u1=0.5,u2=0.3,l=0.2,r=1.0,mi=200,tk=1,nt=10:
   Genetic (ALL): 10 me

## 📋 Missing Data and Skipped Experiments Report

This cell provides a comprehensive summary of any experiments that were skipped due to missing or invalid data during metrics processing.

In [4]:
# Generate comprehensive missing data report
print("📋 MISSING DATA AND SKIPPED EXPERIMENTS REPORT")
print("=" * 60)

# Metrics Aggregation Report
print("\n📊 METRICS AGGREGATION - SKIPPED EXPERIMENTS")
print("-" * 50)
if SKIPPED_METRICS:
    print(f"📊 Total skipped experiments: {len(SKIPPED_METRICS)}\n")
    
    for item in SKIPPED_METRICS:
        print(f"🔸 {item['dataset']} + {item['model']}")
        print(f"   Reason: {item['reason']}")
        print(f"   Path: {item['experiment_path']}")
        print()
else:
    print("✅ No metric aggregation experiments were skipped!")

# Missing Strategies Summary
print("\n🎯 MISSING OPTIMIZATION STRATEGIES BY DATASET-MODEL")
print("-" * 50)
if MISSING_DATA_SUMMARY:
    print(f"📊 Dataset-model combinations with missing strategies: {len(MISSING_DATA_SUMMARY)}\n")
    
    for key, info in sorted(MISSING_DATA_SUMMARY.items()):
        print(f"🔸 {info['dataset']} + {info['model']}")
        print(f"   Missing strategies: {len(info['missing_strategies'])}")
        for strategy in info['missing_strategies']:
            print(f"   - {strategy}")
        print()
else:
    print("✅ All expected optimization strategies found for processed dataset-model combinations!")

# Overall Summary
print("\n📈 OVERALL SUMMARY")
print("-" * 30)
total_metrics_skipped = len(SKIPPED_METRICS)
total_missing_strategies = sum(len(info['missing_strategies']) for info in MISSING_DATA_SUMMARY.values())

print(f"📊 Experiments skipped: {total_metrics_skipped}")
print(f"🎯 Missing strategies: {total_missing_strategies}")

if total_metrics_skipped == 0 and total_missing_strategies == 0:
    print("\n🎉 Perfect! No missing data or skipped experiments detected!")
else:
    print(f"\n⚠️  Total issues detected: {total_metrics_skipped + total_missing_strategies}")
    print("\nℹ️  This is likely due to:")
    print("   • Experiments that failed during execution")
    print("   • Parameter settings that caused optimization to fail")
    print("   • Hardware/time limitations during HPC runs")
    print("   • Strategy-specific compatibility issues with certain datasets/models")

# Save detailed report to file
if any([SKIPPED_METRICS, MISSING_DATA_SUMMARY]):
    report_path = os.path.join(CSV_OUTPUT_DIR, "missing_data_report.json")
    report_data = {
        'skipped_metrics': SKIPPED_METRICS,
        'missing_data_summary': MISSING_DATA_SUMMARY,
        'generated_at': pd.Timestamp.now().isoformat()
    }
    
    os.makedirs(CSV_OUTPUT_DIR, exist_ok=True)
    with open(report_path, 'w') as f:
        json.dump(report_data, f, indent=2)
    print(f"\n💾 Detailed report saved to: {report_path}")

print("\n" + "=" * 60)

📋 MISSING DATA AND SKIPPED EXPERIMENTS REPORT

📊 METRICS AGGREGATION - SKIPPED EXPERIMENTS
--------------------------------------------------
✅ No metric aggregation experiments were skipped!

🎯 MISSING OPTIMIZATION STRATEGIES BY DATASET-MODEL
--------------------------------------------------
✅ All expected optimization strategies found for processed dataset-model combinations!

📈 OVERALL SUMMARY
------------------------------
📊 Experiments skipped: 0
🎯 Missing strategies: 0

🎉 Perfect! No missing data or skipped experiments detected!

